# Семинар 8. Matplotlib и Seaborn: базовые графики и оформление

Ноутбук занятия: материал разбирается вместе с преподавателем, ячейки выполняются сверху вниз.

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-08.ipynb`, например `2MP9-Ivanov-Ivan-seminar-08.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Картинки сохраняются в папку `notebooks/img/`, она создаётся первой ячейкой и в репозиторий не попадает.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-08-tasks.md` с формулировками и ноутбук `seminar-08-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 8».
- Шпаргалка по функциям: `reference/visualization.md`.

## 1. Зачем график и как он устроен

Всё, что будет нарисовано сегодня, вы уже считали: `value_counts`, `groupby`, `crosstab`. График не новая математика, а другой способ показать ту же сводку, и у каждого вопроса свой вид графика:

- «Когда?» — линия: ДТП по годам, месяцам, часам.
- «Сколько чего?» — столбцы: виды ДТП, районы, погода.
- «Как распределено?» — гистограмма и ящик с усами: раненые в ДТП, стаж водителей.
- «Как связаны два признака?» — тепловая карта для двух категорий; связи между числами — на следующем занятии.

Библиотек две. `matplotlib` рисует всё и даёт устройство рисунка: фигура, оси, подписи. `seaborn` поверх него строит статистические графики одной строкой прямо из таблицы. Метод `.plot` у таблиц pandas тоже рисует через matplotlib. Первая ячейка подключает библиотеки, задаёт оформление по умолчанию и создаёт папку для картинок.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')                                         # оформление графиков по умолчанию
os.makedirs('img', exist_ok=True)                                        # папка для картинок
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id', parse_dates=['datetime'])   # ДТП Брянской области
dtp.shape

Сводка числами, знакомая по прошлым занятиям: ДТП по годам.

In [ ]:
by_year = dtp.groupby(dtp['datetime'].dt.year).size()                    # ДТП по годам
by_year

Та же сводка линией. Рисунок в matplotlib устроен из двух частей: **фигура** (`fig`) — лист, на котором рисуют, и **оси** (`ax`) — область с координатами на этом листе. `plt.subplots` создаёт и то и другое; `figsize` задаёт размер листа в дюймах. Метод `.plot` у Series рисует на переданных осях: по X индекс, по Y значения.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))                                   # фигура и оси
by_year.plot(ax=ax, marker='o')                                          # линия по годам

Под ячейкой напечатался и график, и строка `<Axes: xlabel='datetime'>`: это возвращаемое значение последней строки, сами оси. Точка с запятой в конце строки убирает надпись. Ось X подписана именем столбца, заголовка нет: так график читается только тем, кто его нарисовал.

Скелет любого графика в курсе, четыре шага: создать оси, нарисовать, подписать, при необходимости сохранить. Подписи ставятся методами осей: `set_title`, `set_xlabel`, `set_ylabel`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
by_year.plot(ax=ax, marker='o')
ax.set_title('ДТП в Брянской области по годам')                          # заголовок
ax.set_xlabel('Год')                                                     # подпись оси X
ax.set_ylabel('Число ДТП');                                              # подпись оси Y

## 2. Линия: время

Линия годится для того, у чего есть порядок: годы, месяцы, часы. ДТП по месяцам за все годы:

In [ ]:
by_month = dtp.groupby(dtp['datetime'].dt.month).size()                  # ДТП по месяцам за все годы
fig, ax = plt.subplots(figsize=(8, 4))
by_month.plot(ax=ax, marker='o')
ax.set_title('ДТП по месяцам, 2015–2025')
ax.set_xlabel('Месяц')
ax.set_ylabel('Число ДТП');

Пик в августе, 1 050 ДТП, минимум в феврале, 595. Сводка для линии должна идти в порядке периодов: `groupby` сортирует ключи сам, а `value_counts` ставит частые значения первыми, и линия по нему получается зигзагом; после `value_counts` нужен `sort_index`.

Две линии на одних осях: каждая рисуется своим вызовом с параметром `label`, легенду собирает `ax.legend()`. Все ДТП и ДТП без лёгких, то есть тяжёлые и с погибшими:

In [ ]:
serious = dtp[dtp['severity'] != 'Легкий']                               # ДТП без лёгких
fig, ax = plt.subplots(figsize=(8, 4))
by_year.plot(ax=ax, marker='o', label='Все ДТП')
serious.groupby(serious['datetime'].dt.year).size().plot(ax=ax, marker='o', label='Без лёгких')
ax.legend()                                                              # легенда из label
ax.set_title('ДТП по годам: все и без лёгких')
ax.set_xlabel('Год')
ax.set_ylabel('Число ДТП');

Верхняя линия падает втрое, с 1 392 до 465, нижняя вдвое, с 816 до 419. Разница между линиями — лёгкие ДТП, и в 2020 году линии сливаются: лёгких ДТП в таблице за этот год нет ни одного. Падение числа ДТП в таблице во многом эффект учёта, и два ряда на одном графике показывают это лучше любой таблицы.

Линия соединяет соседние точки, поэтому для столбца с текстом она не имеет смысла: у видов ДТП нет соседей. pandas отказывается рисовать линию по тексту; ошибка в следующей ячейке намеренная.

In [ ]:
dtp['category'].plot()                                                   # линия по тексту: ошибка

## 3. Столбцы: категории

Для категорий нужны столбцы. Виды ДТП: сводка `value_counts`, шесть самых частых, метод `.plot.barh` — горизонтальные столбцы, потому что названия длинные. pandas рисует первое значение внизу, а самый частый вид хочется видеть сверху: `ax.invert_yaxis()` разворачивает ось.

In [ ]:
top6 = dtp['category'].value_counts().head(6)                            # шесть самых частых видов
fig, ax = plt.subplots(figsize=(8, 4))
top6.plot.barh(ax=ax)                                                    # горизонтальные столбцы
ax.invert_yaxis()                                                        # самый частый сверху
ax.set_title('Шесть самых частых видов ДТП')
ax.set_xlabel('Число ДТП')
ax.set_ylabel('');

Столкновений 3 994, наездов на пешеходов 2 697; дальше съезд с дороги, 881, и три вида по 547–625. Всего видов 17, остальные одиннадцать вместе дают меньше, чем один съезд с дороги.

seaborn делает то же прямо из сырой таблицы, без `value_counts`: `countplot` считает сам. Параметр `order` задаёт порядок столбцов; здесь он берётся из готовой сводки, чтобы столбцы шли по убыванию.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=dtp, y='category', order=top6.index, ax=ax)           # то же из сырой таблицы
ax.set_title('Шесть самых частых видов ДТП')
ax.set_xlabel('Число ДТП')
ax.set_ylabel('');

Столбец может показывать не число ДТП, а среднее по группе: `barplot` считает среднее указанного столбца и рисует усы — доверительный интервал среднего, тем шире, чем меньше группа.

In [ ]:
frequent = dtp[dtp['category'].isin(top6.index)]                          # ДТП шести частых видов
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=frequent, x='injured_count', y='category', order=top6.index, ax=ax)   # среднее число раненых
ax.set_title('Среднее число раненых в ДТП по видам')
ax.set_xlabel('Раненых на одно ДТП')
ax.set_ylabel('');

В столкновении раненых в среднем 1,47, при съезде с дороги, наезде на препятствие и опрокидывании около 1,24, а при наезде на пешехода или велосипедиста 0,88–0,89: в столкновении страдают люди из двух машин, а сбитый пешеход в части ДТП не ранен, а погиб, и в число раненых не входит.

Столбцы с накоплением показывают состав: доля каждой тяжести внутри года. Сводка — `crosstab` с `normalize='index'`, доли внутри строки; порядок слоёв задаёт список столбцов; `stacked=True` ставит слои друг на друга; `rot=0` не поворачивает подписи годов. Тяжесть упорядочена, поэтому слои раскрашены оттенками одного цвета от светлого к тёмному, а не случайными цветами палитры.

In [ ]:
shares = pd.crosstab(dtp['datetime'].dt.year, dtp['severity'], normalize='index')   # доли тяжести по годам
layers = shares[['Легкий', 'Тяжёлый', 'С погибшими']]                     # слои снизу вверх
fig, ax = plt.subplots(figsize=(8, 4))
layers.plot(kind='bar', stacked=True, rot=0, ax=ax,
            color=['#c6dbef', '#6baed6', '#08519c'])                    # три оттенка синего
ax.set_title('Тяжесть ДТП по годам')
ax.set_xlabel('Год')
ax.set_ylabel('Доля ДТП');

До 2019 года лёгкие ДТП составляли 41–50 % таблицы, в 2020-м их нет совсем, дальше 10–23 %. Доля ДТП с погибшими при этом выросла с 0,10–0,14 до 0,14–0,22 не потому, что дороги стали опаснее, а потому, что из знаменателя ушли лёгкие ДТП. Читая доли, всегда спрашивайте, что стоит в знаменателе.

## 4. Гистограмма и ящик с усами: распределение

Гистограмма показывает, как значения распределены по шкале. Для целых значений, таких как час суток, нужен столбец на каждое целое: `discrete=True` в `sns.histplot`. Функция принимает либо `x=` с Series, либо `data=` с таблицей и имя столбца.

In [ ]:
hours = dtp['datetime'].dt.hour                                          # час каждого ДТП
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x=hours, discrete=True, ax=ax)                              # столбец на каждый час
ax.set_title('ДТП по часам суток')
ax.set_xlabel('Час')
ax.set_ylabel('Число ДТП');

Пик в 18 часов, 798 ДТП, минимум в 4 часа, 107. Число бинов — главная настройка гистограммы, и от неё зависит вывод: те же часы с десятью бинами.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x=hours, bins=10, ax=ax)                                    # те же часы, десять бинов
ax.set_title('ДТП по часам суток: десять бинов')
ax.set_xlabel('Час')
ax.set_ylabel('Число ДТП');

С десятью бинами каждый столбец охватывает 2,3 часа, границы бинов не совпадают с целыми часами, и самым высоким оказывается столбец 14–16 часов: вечерний пик 17–19 часов граница бина разрезала пополам. Бины для целых величин — по единице; для непрерывных величин, таких как сумма иска, число бинов подбирается так, чтобы форма распределения была видна, а случайные зубцы нет.

seaborn с `data=` ищет столбец по имени, и столбца `hour` в таблице пока нет; ошибка в следующей ячейке намеренная.

In [ ]:
sns.histplot(data=dtp, x='hour')                                         # столбца hour нет: ошибка

Сообщение называет имя, которого нет в `data`. Выход — записать час столбцом таблицы, тогда форма с `data=` работает; она удобна, когда на график идут несколько столбцов одной таблицы. Раненых в одном ДТП:

In [ ]:
dtp['hour'] = hours                                                      # час как столбец таблицы
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=dtp, x='injured_count', discrete=True, ax=ax)          # раненых в ДТП
ax.set_title('Раненых в одном ДТП')
ax.set_xlabel('Раненых')
ax.set_ylabel('Число ДТП');

Почти во всех ДТП один раненый, хвост тянется до 14; значений 10, 12 и 13 не было, и на их месте пробелы: гистограмма хранит шкалу. Столбчатая диаграмма по `value_counts` эти пробелы съела бы.

Ящик с усами сравнивает распределения по группам. Стаж водителей по тяжести ДТП: таблица участников, коды 95, 96 и 99 заменяются пропуском, водители соединяются со своими ДТП.

In [ ]:
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')   # участники ДТП
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды стажа пропусками
drivers = people[people['role'] == 'Водитель'].merge(dtp, left_on='dtp_id', right_index=True)   # водители с их ДТП
drivers.shape

In [ ]:
order = ['Легкий', 'Тяжёлый', 'С погибшими']                             # порядок ящиков
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=drivers, x='severity', y='experience_clean', order=order, ax=ax)   # ящики стажа по тяжести
ax.set_title('Стаж водителей по тяжести ДТП')
ax.set_xlabel('Тяжесть ДТП')
ax.set_ylabel('Стаж, лет');

Как читать ящик: линия внутри — медиана, границы ящика — первый и третий квартили, между ними половина водителей; усы тянутся до полутора размахов ящика, точки за усами — отдельные значения. Медианы 12, 13 и 14 лет, ящики почти одинаковые: по стажу водители трёх групп не различаются. `order` задаёт порядок групп и убирает малые: группу «Без пострадавших (нет данных)» из одного водителя рисовать не стоит.

## 5. Тепловая карта: две категории

Сводная таблица двух категорий читается глазами плохо, когда в ней 7 × 24 клетки. Тепловая карта красит клетки по числу: чем темнее, тем больше. Сводка — `crosstab`; имена осей карта берёт из имён Series, поэтому им даётся имя через `rename`, иначе на обеих осях будет `datetime`.

In [ ]:
table = pd.crosstab(dtp['datetime'].dt.dayofweek.rename('День недели'), dtp['hour'].rename('Час'))   # день недели × час
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(table, cmap='Blues', ax=ax)                                  # чем темнее, тем больше
ax.set_title('ДТП по дням недели и часам, понедельник 0');

Самая тёмная клетка — пятница, 18 часов, 129 ДТП; самая светлая — 7. Цвет один, от светлого к тёмному: так карта читается как «больше — меньше». Радужные палитры для чисел не годятся: у радуги нет порядка.

Когда клеток мало, в них можно вписать числа: `annot=True`, формат `fmt='d'` для целых.

In [ ]:
small = pd.crosstab(dtp['datetime'].dt.dayofweek.rename('День недели'), dtp['severity'].rename('Тяжесть'))   # день недели × тяжесть
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(small, cmap='Blues', annot=True, fmt='d', ax=ax)             # числа в клетках
ax.set_title('Тяжесть ДТП по дням недели');

## 6. Рассеяние как проверка данных

Диаграмма рассеяния ставит точку на каждую строку по двум числам. У ДТП есть координаты, и точки по долготе и широте — это карта без подложки. `ax.scatter` рисует точки; `s` — размер, `alpha` — прозрачность, чтобы десять тысяч точек не слились в пятно.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(dtp['lon'], dtp['lat'], s=2, alpha=0.3)                       # точки: долгота, широта
ax.set_title('Координаты ДТП: как записаны')
ax.set_xlabel('Долгота')
ax.set_ylabel('Широта');

Область должна была занять весь рисунок, а заняла угол. Точка в начале координат — ДТП с нулевой широтой и долготой. Точки правее и ниже области, около 52 градусов по X и 33 по Y, — записи, где широта и долгота переставлены местами. Точки далеко справа и сверху — координаты других регионов. `describe` показал бы это числами: минимум 0 и максимум 103 градуса долготы, но на графике ошибка видна с первого взгляда. Отбор рамки области двумя масками убирает мусор; рамка взята с небольшим запасом по границам области:

In [ ]:
inside = dtp[dtp['lat'].between(51.8, 54.2) & dtp['lon'].between(31, 35.5)]   # ДТП внутри рамки области
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(inside['lon'], inside['lat'], s=2, alpha=0.3)
ax.set_title('Координаты ДТП внутри области')
ax.set_xlabel('Долгота')
ax.set_ylabel('Широта')
(dtp['lat'] == 0).sum(), len(dtp) - len(inside)                          # с нулями, за рамкой

За рамкой осталось 367 ДТП: 224 с нулевыми координатами и 143 с координатами за пределами рамки; у 41 из них широта и долгота переставлены, остальные попадают в соседние и дальние регионы. Внутри рамки точки складываются в дороги и города: тёмное пятно — Брянск, лучи от него — трассы. График здесь не украшение отчёта, а способ найти ошибку в данных. Настоящая карта с подложкой — на следующем занятии.

## 7. Оформление, сохранение, итоги

Правила оформления графика в отчёте:

- Заголовок называет, что показано, а не «График 1».
- Оси подписаны, единицы названы; подписи по-русски.
- Столбцы отсортированы, самый большой первым; длинные названия — горизонтальные столбцы.
- Один цвет, пока цвет не несёт смысла; несколько цветов — только для нескольких рядов, и тогда нужна легенда.
- Сетка бледная, ось значений начинается с нуля, размер рисунка под страницу.

Круговая диаграмма нарушает третье правило сама: доли на ней сравниваются на глаз по углам. Один и тот же набор значений двумя способами, на двух панелях одной фигуры: `plt.subplots(1, 2)` возвращает две оси в массиве `axes`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))                          # две панели в ряд
top6.plot.pie(ax=axes[0], ylabel='')                                     # круговая: антипример
top6.plot.barh(ax=axes[1])
axes[1].invert_yaxis()
axes[0].set_title('Круговая диаграмма')
axes[1].set_title('Столбцы')
axes[1].set_xlabel('Число ДТП')
axes[1].set_ylabel('')
fig.suptitle('Шесть видов ДТП: два способа показать')                    # общий заголовок
fig.tight_layout();                                                      # панели не наезжают

Слева не видно, чего больше: съездов с дороги или наездов на велосипедистов; справа это видно сразу, а заодно видно, во сколько раз. Круговая диаграмма в курсе больше не появится.

Две панели по делу: линия по месяцам и гистограмма по часам, общий заголовок `fig.suptitle`, `fig.tight_layout()` раздвигает панели, чтобы подписи не наезжали друг на друга.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_month.plot(ax=axes[0], marker='o')
axes[0].set_title('По месяцам')
axes[0].set_xlabel('Месяц')
axes[0].set_ylabel('Число ДТП')
sns.histplot(x=hours, discrete=True, ax=axes[1])
axes[1].set_title('По часам')
axes[1].set_xlabel('Час')
axes[1].set_ylabel('Число ДТП')
fig.suptitle('Когда происходят ДТП в Брянской области')
fig.tight_layout();

График для отчёта или презентации сохраняется в файл. Ловушка без ошибки: `plt.savefig()` сохраняет «текущую» фигуру, а после показа в ячейке текущей фигуры уже нет, и в файл уходит пустой лист. Следующая ячейка делает именно это; посмотрите на размер файла.

In [ ]:
plt.savefig('img/wrong.png')                                             # текущей фигуры уже нет
os.path.getsize('img/wrong.png')                                         # размер файла в байтах

Пара килобайт — столько весит белый лист. Правильно сохранять фигуру по имени: `fig.savefig`; `dpi=150` даёт разрешение для печати, `bbox_inches='tight'` обрезает пустые поля, чтобы подписи не потерялись. Переменная `fig` из прошлой ячейки жива, и сохранить можно и позже.

In [ ]:
fig.savefig('img/dtp-when.png', dpi=150, bbox_inches='tight')            # фигура из прошлой ячейки
os.path.getsize('img/dtp-when.png')

Файл в десятки килобайт — настоящая картинка; откройте `notebooks/img/dtp-when.png` и убедитесь. В функциях, которые возвращают оси, фигура берётся у осей: `ax.figure.savefig(...)`.

Памятка из пяти вопросов к любому графику, в том числе к графикам этапа 2 проекта:

1. Что показывает график — одно предложение, оно же заголовок.
2. Что по осям и в каких единицах; подписи по-русски.
3. Тот ли тип графика: время линией, категории столбцами, распределение гистограммой или ящиком, две категории тепловой картой.
4. Порядок и масштаб: столбцы отсортированы, ось значений с нуля, бины по смыслу.
5. Что читатель должен вынести — вывод одним предложением под графиком.

## Итоги

Рисунок состоит из фигуры и осей: `fig, ax = plt.subplots(figsize=...)`; любой график строится в четыре шага: создать оси, нарисовать, подписать, сохранить. Сводки pandas рисуются методом `.plot`: линия по периодам, `plot.barh` для категорий, `kind='bar', stacked=True` для долей. seaborn рисует из сырой таблицы: `countplot` и `barplot` для столбцов, `histplot` с `discrete=True` для целых величин, `boxplot` с `order` для сравнения групп, `heatmap` для двух категорий. `ax.scatter` по координатам находит ошибки в данных. Подписи ставят `set_title`, `set_xlabel`, `set_ylabel`; несколько панелей даёт `plt.subplots(1, 2)` с `suptitle` и `tight_layout`; файл сохраняет `fig.savefig` с `dpi` и `bbox_inches='tight'`, а не `plt.savefig` в другой ячейке.

Переходите к упражнениям: откройте `exercises/seminar-08-tasks.ipynb`, формулировки в `exercises/seminar-08-tasks.md`; данные там по Московской области. Функции части Б возвращают оси и принимают параметр `ax`, как функции seaborn: из них в части В собираются панели. На следующем занятии выдаётся этап 2 проекта, где графиков нужно не меньше пяти; необязательная задача этого семинара — три графика по своему датасету.